# Pharmasense — Prophet: Seasonal Baseline Forecaster
**Seasonal decomposition model for malaria + dengue. Interpretable, fast, explainable.**

- No GPU needed (set CPU runtime if you want)
- Downloads data from GitHub automatically

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhi-dev99/agtechathon/blob/main/ml/notebooks/05_train_prophet.ipynb)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'prophet', 'pandas', 'numpy', 'matplotlib', 'scikit-learn', 'seaborn'], check=True)
print('Done.')

In [ ]:
GITHUB_RAW = 'https://raw.githubusercontent.com/abhi-dev99/agtechathon/main/ml/data'
DRIVE_OUTPUT = '/content/drive/MyDrive/pharmasense_models/'
import os
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
# Download data
import os
from google.colab import files
REQUIRED_FILES = ['malaria_weekly.csv.gz', 'epiclim_clean.csv']
missing = [f for f in REQUIRED_FILES if not os.path.exists(f'/content/{f}')]
if missing:
    print('GitHub repository is private. Please upload the following files from your local ml/data/ folder:\n')
    for f in missing:
        print(f'  - {f}')
    print('\n')
    uploaded = files.upload()
    missing = [f for f in REQUIRED_FILES if not os.path.exists(f'/content/{f}')]
    if missing:
        raise FileNotFoundError(f'Still missing: {missing}')
print('All data files are present.')


In [ ]:
import pandas as pd
import numpy as np
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

mal = pd.read_csv('/content/malaria_weekly.csv.gz', parse_dates=['week_start_date'])
mal_state = mal.groupby(['week_start_date','state'])['cases'].sum().reset_index()
mal_state.columns = ['ds','state','cases']

epi = pd.read_csv('/content/epiclim_clean.csv', parse_dates=['week_of_outbreak'])
dengue = epi[epi['Disease'].str.lower().str.contains('dengue', na=False)].copy()
dengue_state = dengue.groupby(['week_of_outbreak','state_ut'])['Cases'].sum().reset_index()
dengue_state.columns = ['ds','state','cases']

print(f'Malaria state-weeks: {len(mal_state):,} | States: {mal_state.state.nunique()}')
print(f'Dengue state-weeks:  {len(dengue_state):,} | States: {dengue_state.state.nunique()}')

In [ ]:
# ── Train Prophet per state-disease, collect metrics ─────────────────────────
results = []
models  = {}

def train_prophet(df_series, state, disease):
    df = df_series[['ds','cases']].rename(columns={'cases':'y'}).copy()
    df['y'] = np.log1p(df['y'].clip(lower=0))
    df = df.dropna().sort_values('ds').reset_index(drop=True)
    if len(df) < 50:
        return None, None
    cutoff = int(len(df) * 0.85)
    train, test = df[:cutoff], df[cutoff:]
    m = Prophet(yearly_seasonality=True, weekly_seasonality=False,
                daily_seasonality=False, seasonality_mode='multiplicative',
                changepoint_prior_scale=0.15)
    m.fit(train)
    future = m.make_future_dataframe(periods=len(test), freq='W')
    forecast = m.predict(future)
    pred_log = forecast['yhat'].iloc[cutoff:].values[:len(test)]
    true_log = test['y'].values
    pred = np.expm1(pred_log)
    true = np.expm1(true_log)
    mask = true > 0
    mae  = mean_absolute_error(true, pred)
    rmse = np.sqrt(mean_squared_error(true, pred))
    mape = np.mean(np.abs((true[mask]-pred[mask])/true[mask]))*100 if mask.sum()>0 else np.nan
    r2   = r2_score(true, pred) if len(true) > 1 else np.nan
    return m, {'state':state, 'disease':disease, 'MAE':mae, 'RMSE':rmse, 'MAPE':mape, 'R2':r2, 'n':len(df)}

# High-burden states for malaria
top_states = mal_state.groupby('state')['cases'].sum().nlargest(10).index.tolist()
for state in top_states:
    sub = mal_state[mal_state['state']==state]
    m, row = train_prophet(sub, state, 'malaria')
    if row: results.append(row); models[f'malaria_{state}'] = m
    print(f'  malaria | {state}: MAPE={row["MAPE"]:.1f}%' if row else f'  malaria | {state}: skipped')

# Dengue states with enough data
dengue_states = dengue_state.groupby('state').size()
dengue_states = dengue_states[dengue_states >= 50].index.tolist()
for state in dengue_states[:8]:
    sub = dengue_state[dengue_state['state']==state]
    m, row = train_prophet(sub, state, 'dengue')
    if row: results.append(row); models[f'dengue_{state}'] = m
    print(f'  dengue  | {state}: MAPE={row["MAPE"]:.1f}%' if row else f'  dengue  | {state}: skipped')

print(f'\nTotal models trained: {len(models)}')

In [ ]:
# ── Metrics summary ──────────────────────────────────────────────────────────
import pandas as pd
metrics_df = pd.DataFrame(results)
print('=' * 60)
print('  PROPHET METRICS SUMMARY')
print('=' * 60)
print(metrics_df.groupby('disease')[['MAE','RMSE','MAPE','R2']].mean().round(2).to_string())
print()
print('Per-state breakdown (malaria):')
print(metrics_df[metrics_df['disease']=='malaria'][['state','MAPE','R2']].to_string(index=False))

In [ ]:
# ── Forecast + components plot for top state ─────────────────────────────────
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

top_state = top_states[0]
m = models.get(f'malaria_{top_state}')
if m:
    future = m.make_future_dataframe(periods=52, freq='W')
    fc = m.predict(future)
    fig1 = m.plot(fc, figsize=(14, 5))
    plt.title(f'Prophet Malaria Forecast — {top_state} (log scale + 1)')
    plt.tight_layout()
    plt.savefig(DRIVE_OUTPUT + f'prophet_malaria_{top_state}.png', dpi=150)
    plt.show()

    fig2 = m.plot_components(fc, figsize=(14, 10))
    plt.suptitle(f'Prophet Seasonality Components — Malaria {top_state}', y=1.01)
    plt.tight_layout()
    plt.savefig(DRIVE_OUTPUT + f'prophet_malaria_{top_state}_components.png', dpi=150)
    plt.show()
    print(f'Plots saved for {top_state}')

In [ ]:
# ── MAPE barplot across states ───────────────────────────────────────────────
import seaborn as sns
fig, ax = plt.subplots(figsize=(12, 5))
mdf = metrics_df.sort_values('MAPE')
colors = ['#2563EB' if d=='malaria' else '#DC2626' for d in mdf['disease']]
ax.barh(mdf['state'] + ' (' + mdf['disease'] + ')', mdf['MAPE'], color=colors)
ax.axvline(20, color='orange', linestyle='--', label='20% threshold')
ax.set_xlabel('MAPE (%)')
ax.set_title('Prophet Forecast Accuracy by State-Disease (lower = better)')
ax.legend()
plt.tight_layout()
plt.savefig(DRIVE_OUTPUT + 'prophet_mape_by_state.png', dpi=150)
plt.show()

In [ ]:
# ── Save all models ───────────────────────────────────────────────────────────
import pickle, os
from datetime import datetime
artifact = {'models': models, 'metrics': metrics_df.to_dict(orient='records'),
            'trained_at': datetime.now().isoformat()}
out_path = DRIVE_OUTPUT + 'prophet_models.pkl'
with open(out_path, 'wb') as f:
    pickle.dump(artifact, f)
from google.colab import files
files.download(out_path)
print(f'Saved {len(models)} models -> {out_path}')
print(f'Overall MAPE: {metrics_df["MAPE"].mean():.1f}%  |  R2: {metrics_df["R2"].mean():.3f}')